In [37]:
import pandas as pd
from datetime import datetime

In [38]:
class TaskPriorityModel:
    def calculate_days_until_due(self, due_date_str):
        if pd.isna(due_date_str):
            return 7  

        try:
            due_date = pd.to_datetime(due_date_str)

            today = datetime.now()
            delta = due_date - today
            return max(0, delta.days)
        except Exception:
            return 7
            
    def predict_priority(self, row_dict):
        due_str = row_dict.get('deadline')
        difficulty = int(row_dict.get('difficulty'))
        weight = int(row_dict.get('weight'))

        days_until_due = self.calculate_days_until_due(due_str)

        urgency_score = max(0, 30 - days_until_due) * 50
        difficulty_score = difficulty * 1
        weight_score = weight * 1

        return urgency_score + difficulty_score + weight_score

    def predict_batch_df(self, df, normalize=False):
        records = df.to_dict(orient='records')
        raw_scores = [self.predict_priority(r) for r in records]

        if not normalize:
            return raw_scores  

        total = sum(raw_scores)
        if total > 0:
            normalized = [round(score / total * 100, 2) for score in raw_scores]
        else:
            normalized = [0 for _ in raw_scores]

        return normalized

In [39]:
df = pd.read_csv('tasks.csv')
df

,name,difficulty,weight,deadline
0,Essay,4,8,12/5/2025
1,Presentation,3,6,12/1/2025
2,Reminder,1,3,12/20/2025


In [40]:
df['deadline'] = pd.to_datetime(df['deadline'])
df

,name,difficulty,weight,deadline
0,Essay,4,8,2025-12-05
1,Presentation,3,6,2025-12-01
2,Reminder,1,3,2025-12-20


In [42]:
df['priority_score'] = model.predict_batch_df(df)

df = df.sort_values('priority_score', ascending=False).reset_index(drop=True)
df

,name,difficulty,weight,deadline,priority_score
0,Presentation,3,6,2025-12-01,1509
1,Essay,4,8,2025-12-05,1312
2,Reminder,1,3,2025-12-20,554


In [43]:
new_task = {
    'name': 'Quiz',
    'difficulty': 2,
    'weight': 5,
    'deadline': '2025-12-01'
}

df = pd.concat([df, pd.DataFrame([new_task])], ignore_index=True)

In [44]:
model = TaskPriorityModel()

df['priority_score'] = model.predict_batch_df(df, normalize=False)

df = df.sort_values('priority_score', ascending=False).reset_index(drop=True)
df

,name,difficulty,weight,deadline,priority_score
0,Presentation,3,6,2025-12-01 00:00:00,1509
1,Quiz,2,5,2025-12-01,1507
2,Essay,4,8,2025-12-05 00:00:00,1312
3,Reminder,1,3,2025-12-20 00:00:00,554
